# Putting the answer in the prompt

MichAl Academy, unit 5.3.

Run each cell with **Shift+Enter**.

A model knows what was in its training data. Retrieval is the machinery for
answering questions about everything else: find the passage that holds the
answer, put it in the prompt, and let the model read rather than remember.

This notebook builds the whole thing in a few dozen lines, on a corpus written
so that **no model can know any of it**.


In [ ]:
import re
import time
import warnings
from collections import Counter

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()


def chat(messages):
    return tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate(prompt, n=24):
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


## A corpus nobody has read

Twenty-four sentences of documentation for a product line that does not exist.
The appliances, the firmware branches, the telemetry format and the support
tiers are all invented.

That is deliberate and it is the only way this measurement means anything. If
the corpus were about Linux, a right answer might come from the model's training
data, and "retrieval helped" would be unprovable. Here a right answer can only
have come from the passage we put in the prompt.


In [ ]:
PASSAGES = [
    "The Kestrel KX-7712 appliance ships with 48 gigabytes of memory and two power supplies.",
    "The Kestrel KX-7712 draws 310 watts under full load and 95 watts at idle.",
    "Firmware branch 9.4 for the Kestrel line adds support for the Harrier telemetry format.",
    "Firmware branch 9.4 removes the legacy Osprey configuration importer.",
    "The Kestrel KX-3300 is the desk-side model and has a single power supply.",
    "The Kestrel KX-3300 holds 16 gigabytes of memory and cannot be expanded.",
    "Kestrel appliances listen for management traffic on TCP port 8443 by default.",
    "The management port on a Kestrel can be moved, but not to any port below 1024.",
    "A Kestrel cluster requires an odd number of members, and the supported sizes are three, five and seven.",
    "A Kestrel cluster loses write availability when more than half its members are unreachable.",
    "The Harrier telemetry format writes one record per flow and compresses at roughly four to one.",
    "Harrier records carry a sixteen byte flow identifier and a millisecond timestamp.",
    "The Osprey configuration importer was deprecated in firmware 9.2 and removed in 9.4.",
    "Configurations exported by Osprey can be converted with the kestrel-convert utility.",
    "The kestrel-convert utility refuses to run on a configuration larger than 64 megabytes.",
    "Support contract tier Gold includes four hour hardware replacement on the Kestrel line.",
    "Support contract tier Silver includes next business day hardware replacement.",
    "The Kestrel warranty covers power supplies for five years and fans for three.",
    "Kestrel appliances log to a ring buffer of 2 gigabytes before overwriting the oldest entries.",
    "Setting the log level to verbose on a Kestrel fills the ring buffer in about six hours.",
    "The Kestrel KX-7712 was released in March 2024 and the KX-3300 followed in September 2024.",
    "Kestrel firmware branch 9.3 is supported until the end of 2026.",
    "A Kestrel appliance performs a configuration backup every night at 02:00 local time.",
    "Kestrel nightly backups are kept for thirty days unless a retention policy says otherwise.",
]

# Each question is answered by exactly one passage, named by its index.
QUESTIONS = [
    ("How much memory does the Kestrel KX-7712 have?", "48", 0),
    ("How many watts does a KX-7712 draw under full load?", "310", 1),
    ("Which telemetry format does firmware 9.4 add support for?", "harrier", 2),
    ("How many power supplies does the KX-3300 have?", "one", 4),
    ("Which TCP port do Kestrel appliances use for management?", "8443", 6),
    ("What cluster sizes are supported?", "three", 8),
    ("How big is the flow identifier in a Harrier record?", "sixteen", 11),
    ("In which firmware version was the Osprey importer removed?", "9.4", 12),
    ("What is the size limit for kestrel-convert?", "64", 14),
    ("What replacement time does the Gold tier include?", "four", 15),
    ("How long are power supplies covered by the warranty?", "five", 17),
    ("At what time does a Kestrel run its nightly backup?", "02:00", 22),
]

print(f"{len(PASSAGES)} passages, {len(QUESTIONS)} questions")


def build(question, note=None):
    if note is None:
        return chat([{"role": "user", "content": question}])
    return chat([{"role": "user", "content":
                  f"Answer the question using only the note below.\n\n"
                  f"Note: {note}\n\nQuestion: {question}"}])


def answered(question, want, note=None):
    return want.lower() in generate(build(question, note)).lower()


## Asking with nothing to read


In [ ]:
closed = sum(answered(q, want) for q, want, _ in QUESTIONS)
print(f"closed book                  {closed}/12")

ceiling = sum(answered(q, want, PASSAGES[gold]) for q, want, gold in QUESTIONS)
print(f"the right passage, handed over {ceiling}/12")


**Zero against ten.** The first number is the point of the invented corpus: the
model cannot answer, and no prompt wording will fix that, because the facts were
never in its training data.

The second number is the ceiling for everything below. Even reading the right
sentence, this model gets two of the twelve wrong, so a retrieval system that
was perfect would still score 10 here. **Keep the two apart**: whether the right
passage was found, and whether the answer was then right, are different failures
with different fixes.


## Three ways to find the passage

**tf-idf** scores a passage by the words it shares with the question, weighting
rare words more heavily. **BM25** does the same with two corrections that matter
in practice: repeated words count for progressively less, and long passages are
penalised for having more chances to match. **Embeddings** compare meaning
rather than words, by turning both question and passage into vectors and taking
the cosine between them, as in unit 1.7.3.


In [ ]:
tfidf = TfidfVectorizer().fit(PASSAGES)
P_TFIDF = tfidf.transform(PASSAGES)


def retrieve_tfidf(q):
    return int((tfidf.transform([q]) @ P_TFIDF.T).toarray()[0].argmax())


def tokenise(text):
    return re.findall(r"[a-z0-9.:-]+", text.lower())


def bm25_factory(docs):
    """BM25 in a dozen lines. k1 controls how fast repeats stop counting,
    b how hard long documents are penalised; these are the usual defaults."""
    toks = [tokenise(d) for d in docs]
    df = Counter(w for d in toks for w in set(d))
    avgdl = sum(len(d) for d in toks) / len(toks)
    n = len(toks)

    def score(q, k1=1.5, b=0.75):
        out = np.zeros(n)
        for w in tokenise(q):
            if w not in df:
                continue
            idf = np.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
            for i, d in enumerate(toks):
                f = d.count(w)
                if f:
                    out[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        return out
    return score


bm25 = bm25_factory(PASSAGES)


@torch.no_grad()
def embed(text):
    """No embedding model is in this image, so the language model's own hidden
    states are averaged into one vector. It works, and it is weak: see below."""
    ids = tok(text, return_tensors="pt").input_ids
    v = model.model(ids).last_hidden_state[0].mean(0)
    return (v / v.norm()).numpy()


EMB = np.stack([embed(p) for p in PASSAGES])
print("passages embedded:", EMB.shape)


In [ ]:
def run(retrieve, label):
    hit = ok = 0
    for q, want, gold in QUESTIONS:
        i = retrieve(q)
        hit += i == gold
        ok += answered(q, want, PASSAGES[i])
    print(f"{label:<28}{hit:>6}/12{ok:>9}/12")


print(f"{'':<28}{'found it':>12}{'answered':>11}")
run(retrieve_tfidf, "tf-idf")
run(lambda q: int(bm25(q).argmax()), "BM25")
run(lambda q: int((EMB @ embed(q)).argmax()), "embeddings")


**Retrieval is the whole difference between 0 and 9.** Two words of machinery,
a scorer and a top-1 lookup, and the model answers questions about a product
that does not exist.

**The embedding row is the weak one, and the reason is this notebook rather than
embeddings.** A real system uses a model trained specifically to place similar
texts near each other. There is no such model in this image, so the vectors here
are the language model's own hidden states averaged together, which is a rough
approximation and scores like one. What transfers is the mechanism, not the
ranking.


## Chunk size

Real documents are longer than one sentence, so something has to decide how much
text goes in each unit that gets retrieved. Glue the same corpus into chunks of
1, 3, 6 and 24 sentences and measure all three things that move.


In [ ]:
print(f"{'sentences per chunk':<22}{'chunks':>8}{'found it':>10}{'answered':>10}{'tokens':>9}")
for size in (1, 3, 6, 24):
    chunks = [" ".join(PASSAGES[i:i + size]) for i in range(0, len(PASSAGES), size)]
    owner = {i: i // size for i in range(len(PASSAGES))}
    score = bm25_factory(chunks)
    hit = ok = 0
    used = []
    for q, want, gold in QUESTIONS:
        i = int(score(q).argmax())
        hit += i == owner[gold]
        prompt = build(q, chunks[i])
        used.append(tok(prompt, return_tensors="pt").input_ids.shape[1])
        ok += want.lower() in generate(prompt).lower()
    print(f"{size:<22}{len(chunks):>8}{hit:>8}/12{ok:>8}/12{int(np.mean(used)):>9}")


**Finding it gets easier and answering gets harder.** The last row is the
"just put the whole document in the prompt" instinct: retrieval cannot fail,
because there is only one chunk, and the model then scores worse than it did
reading three sentences, for five times the tokens.

That is the trade chunking exists to manage, and note which side of it pays.
Chunking is not there to make retrieval work. It is there to keep the reader
from drowning.


The first two rows of that table answer equally well, 9 and 9, while the second
finds two more passages. Where did the two go? Look at the questions one at a
time rather than at the totals.


In [ ]:
bm_one = bm25_factory(PASSAGES)
chunks3 = [" ".join(PASSAGES[i:i + 3]) for i in range(0, len(PASSAGES), 3)]
bm_three = bm25_factory(chunks3)

print(f"{'q':<3}{'found at 1':>11}{'answered at 1':>15}{'answered at 3':>15}{'ceiling':>9}")
for n, (q, want, gold) in enumerate(QUESTIONS):
    i1 = int(bm_one(q).argmax())
    row = (i1 == gold,
           want.lower() in generate(build(q, PASSAGES[i1])).lower(),
           want.lower() in generate(build(q, chunks3[int(bm_three(q).argmax())])).lower(),
           want.lower() in generate(build(q, PASSAGES[gold])).lower())
    flag = "  <-- changed" if row[1] != row[2] else ""
    print(f"{n:<3}{str(row[0]):>11}{str(row[1]):>15}{str(row[2]):>15}{str(row[3]):>9}{flag}")


**The two totals are equal and the questions underneath are not.** Going from
one sentence to three **recovered** the question whose passage the small chunks
never retrieved, and **broke** a different question that the small chunks had
always answered: more context around the right sentence was enough to lose it.

Two of the twelve are beyond this model whatever it is shown, which is the
ceiling from the first cell, and one of those is also one that retrieval missed.
So the same 9 is not the same nine questions, and a total that does not move can
still be hiding two changes in opposite directions.


## What keyword search does that vectors do not

Two passages that differ only in a model code, and four questions that each name
one of them.


In [ ]:
PAIR = [
    "The Kestrel KX-7712 appliance ships with 48 gigabytes of memory and two power supplies.",
    "The Kestrel KX-3300 appliance ships with 16 gigabytes of memory and one power supply.",
]
CODE_Q = [("How much memory does the KX-7712 have?", 0),
          ("How much memory does the KX-3300 have?", 1),
          ("How many power supplies does the KX-7712 have?", 0),
          ("How many power supplies does the KX-3300 have?", 1)]

pair_bm25 = bm25_factory(PAIR)
pair_emb = np.stack([embed(p) for p in PAIR])
hb = sum(int(pair_bm25(q).argmax()) == gold for q, gold in CODE_Q)
he = sum(int((pair_emb @ embed(q)).argmax()) == gold for q, gold in CODE_Q)
print(f"BM25 picks the right passage        {hb}/4")
print(f"embeddings pick the right passage   {he}/4")


**An identifier is not a meaning.** `KX-7712` and `KX-3300` are near-identical
as text and describe near-identical products, so a similarity score has almost
nothing to separate them. A keyword index has everything to separate them,
because the two strings are simply different.

This is why production search is usually **hybrid**: run both and combine the
rankings. The standard combination is reciprocal rank fusion, which adds
1/(60 + rank) from each list and sorts by the total, so a passage ranked highly
by either method survives.


In [ ]:
def fuse(q, k=60):
    ranks_b = list(np.argsort(-bm25(q)))
    ranks_e = list(np.argsort(-(EMB @ embed(q))))
    total = {}
    for lst in (ranks_b, ranks_e):
        for rank, idx in enumerate(lst):
            total[idx] = total.get(idx, 0) + 1 / (k + rank + 1)
    return max(total, key=total.get)


run(fuse, "hybrid, rank fusion")


## Reranking

Retrieval has to be cheap, because it scores every passage in the corpus.
Reranking is a second, more expensive pass over the handful that survived.

The method here is **query likelihood**: put a candidate passage in front of the
model and measure how probable the question becomes. A passage that makes the
question unsurprising is a passage that answers it. A production reranker is a
model trained for this job rather than borrowed from generation.


In [ ]:
@torch.no_grad()
def query_likelihood(passage, question):
    p_ids = tok(f"Passage: {passage}\nQuestion:", return_tensors="pt").input_ids
    q_ids = tok(" " + question, return_tensors="pt").input_ids
    ids = torch.cat([p_ids, q_ids], dim=1)
    logits = model(ids).logits[0, :-1]
    targets = ids[0, 1:]
    logp = torch.log_softmax(logits, -1)[range(len(targets)), targets]
    return float(logp[p_ids.shape[1] - 1:].mean())


started = time.time()
top1 = reranked = in_top5 = 0
for q, want, gold in QUESTIONS:
    order = list(np.argsort(-bm25(q))[:5])
    top1 += order[0] == gold
    in_top5 += gold in order
    reranked += max(order, key=lambda i: query_likelihood(PASSAGES[i], q)) == gold
print(f"BM25 top 1                   {top1}/12")
print(f"gold passage in the top 5    {in_top5}/12")
print(f"after reranking the top 5    {reranked}/12   ({time.time() - started:.0f}s)")


**A null, and it is worth reading carefully.** There was exactly one place to
gain: eleven of twelve questions had the right passage somewhere in the top five,
and ten already had it first. Reranking did not find the twelfth.

What this shows is the shape of the technique rather than its value. Reranking
can only reorder what retrieval handed it, so **its ceiling is recall, not
accuracy**: on this corpus that ceiling is 11 of 12, and one wrong order was not
enough to demonstrate anything. At production scale, where retrieval returns
twenty candidates out of millions, Anthropic report reranking taking a 49%
reduction in failed retrievals to 67%.


## What this unit measured

- **Nothing in the weights, everything in the prompt.** Closed book 0 of 12,
  with retrieval 9, with perfect retrieval 10.
- **Two scores, not one.** Finding the passage and answering from it fail
  separately, and the second one has a ceiling the first cannot lift.
- **Chunk size trades three things at once**: retrieval gets easier, the answer
  gets harder, and the prompt gets more expensive.
- **Keyword matching keeps what embeddings lose**, which is the exact string,
  and identifiers are exact strings.
- **Reranking cannot beat recall**, so it is worth what the candidate list is
  worth.
